# Part 3: Granger causality (ds002680 go/no-go)

Does one channel's past help predict another channel, beyond that channel's own past?

For target channel **y** and source channel **x**, pooled over trials:

| model | predictors of y[t] |
|---|---|
| restricted | y[t−1 … t−p] (its own past) |
| full | y[t−1 … t−p] **and** x[t−1 … t−p] |

**GC(x → y) = ln(RSS_restricted / RSS_full)**. RSS is the residual sum of squares, the prediction error left over.
- GC = 0: x's past adds nothing. GC > 0: the full model predicts better.
- GC is never negative (more predictors never fit worse), so "adding value" needs a test: an **F-test** on the extra p coefficients, and a **permutation test** that shuffles which trial x comes from.
- On scalp EEG every channel mixes the same sources, so almost every pair passes those tests. **Time-reversed GC (TRGC)** separates mixing from real flow: net flow (x→y minus y→x) forward minus the same on time-reversed data. Mixing gives TRGC ≈ 0; true x → y gives TRGC > 0 (Haufe et al. 2013).

Done per condition (target / distractor) in equal-length pre and post windows. The evoked response is removed first (it is identical in every trial and would look like prediction).

In [ ]:
# re-read edited .py files in src/bids_conn on every cell run
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns

from bids_conn import find_recordings, get_paths, load_raw, make_epochs, stimulus_metadata
from bids_conn.connectivity import crop_window, subsample
from bids_conn.granger import granger_pair, lagged, pairwise_granger, prepare_for_gc

mne.set_log_level("WARNING")

paths = get_paths()
out_dir = paths["results"] / "part3_granger"      # one CSV per recording goes here
out_dir.mkdir(parents=True, exist_ok=True)
print(paths["bids_root"], paths["bids_root"].exists())

## Config

In [ ]:
recs = find_recordings(paths["bids_root"], task="gonogo")
display(recs[["subject", "session", "run", "rec_id"]])
rec_ids = list(recs["rec_id"])        # all recordings; or e.g. rec_ids = rec_ids[:1] to test

# preprocessing and epochs (same as parts 1 and 2)
l_freq, h_freq = 1.0, 40.0
reference      = 'average'
resample_to    = 500
epoch_tmin, epoch_tmax = -0.6, 1.0
chans      = ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
reject_uv  = 100
correct_only = True
conditions = ['target', 'distractor']
equalize_trials = True
seed = 0

# Granger
gc_sfreq  = 250                       # 4 ms per lag: an order of 10-15 covers 40-60 ms of history
windows   = {'pre': (-0.5, -0.01), 'post': (0.01, 0.5)}   # equal length
max_order = 15                        # model order (number of lags) chosen by BIC up to this
n_perm    = 100                       # trial-shuffle permutations per pair (0 = F-test only)
overwrite = False

## One pair, step by step
What `granger_pair` does for one source → target pair in one recording, so the numbers in the results table are not a black box.

In [ ]:
rec = recs.iloc[0]
raw = load_raw(rec.path, l_freq=l_freq, h_freq=h_freq, sfreq=resample_to, reference=reference)
epochs, n_rej = make_epochs(raw, stimulus_metadata(rec.path), tmin=epoch_tmin, tmax=epoch_tmax,
                            reject_uv=reject_uv, reject_picks=chans)
epochs = epochs[epochs.metadata["correct"].to_numpy()].pick(chans).resample(gc_sfreq)
win = crop_window(epochs["target"], {'post': windows['post']})['post']

data = prepare_for_gc(win.get_data())              # (trials, chans, times): evoked removed, z-scored
source, target, order = 'O2', 'P3', 10
x = data[:, win.ch_names.index(source)]            # (trials, times)
y = data[:, win.ch_names.index(target)]

y_now = y[:, order:].reshape(-1)                   # what we predict: every sample after the first `order`
own, other = lagged(y, order), lagged(x, order)    # y's past, x's past: (observations, order) each
print("observations:", len(y_now), "| own-past predictors:", own.shape[1], "| source-past predictors:", other.shape[1])

def rss(X, y_):
    X1 = np.column_stack([np.ones(len(y_)), X])
    beta, *_ = np.linalg.lstsq(X1, y_, rcond=None)
    return np.sum((y_ - X1 @ beta) ** 2)

rss_r, rss_f = rss(own, y_now), rss(np.hstack([own, other]), y_now)
print(f"RSS restricted {rss_r:.1f} | RSS full {rss_f:.1f} | GC = ln(ratio) = {np.log(rss_r / rss_f):.4f}")
print("granger_pair gives:", granger_pair(y, x, order))
del raw

## Pipeline for one recording

In [ ]:
def run_recording(rec):
    """Load one recording, compute Granger causality for every directed pair per condition and window."""
    raw = load_raw(rec.path, l_freq=l_freq, h_freq=h_freq, sfreq=resample_to, reference=reference)
    meta = stimulus_metadata(rec.path)
    epochs, n_rejected = make_epochs(raw, meta, tmin=epoch_tmin, tmax=epoch_tmax,
                                     reject_uv=reject_uv, reject_picks=chans)
    if correct_only:
        epochs = epochs[epochs.metadata["correct"].to_numpy()]
    epochs = epochs.pick(chans).resample(gc_sfreq)

    rng = np.random.default_rng(seed)
    by_cond = {cond: epochs[cond] for cond in conditions}
    if equalize_trials:
        n_min = min(len(ep) for ep in by_cond.values())
        by_cond = {cond: subsample(ep, n_min, rng) for cond, ep in by_cond.items()}

    rows = []
    for cond, ep in by_cond.items():
        for win_name, win in crop_window(ep, windows).items():
            for row in pairwise_granger(win, max_order=max_order, n_perm=n_perm, rng=rng):
                row.update(condition=cond, window=win_name, subject=rec.subject, session=rec.session,
                           run=rec.run, rec_id=rec.rec_id, n_trials=len(ep), n_rejected=n_rejected)
                rows.append(row)
    return rows

## Run all recordings
Each recording's rows are saved as soon as it finishes; a failing recording is reported and skipped. Set `overwrite = True` after changing the config.

In [ ]:
failed = []
todo = recs[recs["rec_id"].isin(rec_ids)]
for i, rec in enumerate(todo.itertuples(), start=1):
    out_file = out_dir / f"{rec.rec_id}.csv"
    if out_file.exists() and not overwrite:
        print(f"[{i}/{len(todo)}] {rec.rec_id}: already done, skipping")
        continue

    try:
        rows = run_recording(rec)
    except Exception as e:
        print(f"[{i}/{len(todo)}] {rec.rec_id}: FAILED - {type(e).__name__}: {e}")
        failed.append(rec.rec_id)
        continue

    pd.DataFrame(rows).to_csv(out_file, index=False)
    print(f"[{i}/{len(todo)}] {rec.rec_id}: done, {len(rows)} rows")

print("failed:", failed)

## Combine
Runs are averaged within subject first.

In [ ]:
df = pd.concat([pd.read_csv(out_dir / f"{r}.csv", dtype={"subject": str, "session": str, "run": str})
                for r in rec_ids if (out_dir / f"{r}.csv").exists()], ignore_index=True)
df.to_csv(paths["results"] / "part3_granger_all_recordings.csv", index=False)

subj = df.groupby(["subject", "condition", "window", "source", "target"], as_index=False)[["gc", "trgc", "net_gc"]].mean()
n_subjects = subj["subject"].nunique()
print(df.shape, "| recordings:", df["rec_id"].nunique(), "| subjects:", n_subjects)

## Does the source add value?
Per condition and window: how many of the 56 directed pairs pass each test. The F-test and permutation test answer "is GC > 0"; a large count there with few TRGC pairs means the 'prediction' is mostly signal mixing.

In [ ]:
summary = df.groupby(["rec_id", "condition", "window"]).agg(
    order=("order", "first"),
    gc_median=("gc", "median"),
    sig_F=("p_F", lambda p: (p < 0.05).sum()),
    **({"sig_perm": ("p_perm", lambda p: (p < 0.05).sum())} if "p_perm" in df else {}),
    trgc_over_0_01=("trgc", lambda t: (t > 0.01).sum()),
    n_pairs=("gc", "size"),
)
summary.round(4)

## Matrices (mean over subjects)
Rows = **source**, columns = **target**. Left: GC. Right: TRGC (red = net source → target flow that survives the time-reversal check).

In [ ]:
def directed_matrix(table, value):
    """source x target matrix of the mean over subjects; diagonal NaN."""
    return table.pivot_table(index="source", columns="target", values=value).reindex(index=chans, columns=chans)


for cond in conditions:
    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    gc_mats = [directed_matrix(subj[(subj.condition == cond) & (subj.window == w)], "gc") for w in windows]
    tr_mats = [directed_matrix(subj[(subj.condition == cond) & (subj.window == w)], "trgc") for w in windows]
    gc_max = max(m.max().max() for m in gc_mats)
    tr_max = max(abs(m).max().max() for m in tr_mats + [tr_mats[1] - tr_mats[0]])

    for c, w in enumerate(windows):
        sns.heatmap(gc_mats[c], ax=axes[0, c], cmap='Blues', vmin=0, vmax=gc_max, annot=True, fmt='.3f', square=True)
        axes[0, c].set_title(f'GC, {w}')
        sns.heatmap(tr_mats[c], ax=axes[1, c], cmap='RdBu_r', vmin=-tr_max, vmax=tr_max, center=0, annot=True, fmt='.3f', square=True)
        axes[1, c].set_title(f'TRGC, {w}')
    sns.heatmap(gc_mats[1] - gc_mats[0], ax=axes[0, 2], cmap='RdBu_r', center=0, annot=True, fmt='.3f', square=True)
    axes[0, 2].set_title('GC, post − pre')
    sns.heatmap(tr_mats[1] - tr_mats[0], ax=axes[1, 2], cmap='RdBu_r', vmin=-tr_max, vmax=tr_max, center=0, annot=True, fmt='.3f', square=True)
    axes[1, 2].set_title('TRGC, post − pre')
    for ax in axes.flat:
        ax.set_ylabel('source'); ax.set_xlabel('target')

    fig.suptitle(f'ds002680, {cond}, mean of {n_subjects} subject(s)')
    plt.tight_layout()
    plt.show()

## Group statistics
Per directed pair: is TRGC > 0 across subjects (Wilcoxon, one-sided), and does it change post vs pre? FDR within each condition × test. Needs ≥ 6 subjects.

In [ ]:
import pingouin as pg
from statsmodels.stats.multitest import multipletests

if n_subjects < 6:
    print(f"Only {n_subjects} subject(s): download more of ds002680 before testing.")
else:
    wide = subj.pivot_table(index=["subject", "condition", "source", "target"], columns="window", values="trgc").reset_index()
    wide["post_minus_pre"] = wide["post"] - wide["pre"]
    results = []
    for (cond, src, tgt), g in wide.groupby(["condition", "source", "target"]):
        for test, col, alt in [("trgc_post > 0", "post", "greater"), ("post - pre", "post_minus_pre", "two-sided")]:
            x = g[col].dropna()
            results.append({"condition": cond, "source": src, "target": tgt, "test": test,
                            "median": x.median(), "p": pg.wilcoxon(x, alternative=alt)["p-val"].iloc[0], "n": len(x)})
    stats = pd.DataFrame(results)
    stats["p_fdr"] = stats.groupby(["condition", "test"])["p"].transform(lambda p: multipletests(p, method="fdr_bh")[1])
    stats.to_csv(paths["results"] / "part3_granger_stats.csv", index=False)
    display(stats.sort_values("p_fdr").head(20))